# AI Image Lab — ComfyUI en Google Colab

Notebook para ejecutar ComfyUI con GPU de Colab, persistir modelos/LoRAs/workflows en Google Drive y acceder a la interfaz desde el navegador.

**Arquitectura:** Colab GPU → ComfyUI → Google Drive (`AI-Image`) → modelos / LoRAs / outputs.


## 1. Configuración
En Colab selecciona **Runtime → Change runtime type → GPU** antes de ejecutar las celdas.

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

print('Entorno preparado.')


In [ ]:
!nvidia-smi

import torch
print('\nPyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), 'GB')


## 2. Google Drive
Los modelos, LoRAs, workflows y resultados permanecerán en Drive aunque la sesión de Colab termine.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/AI-Image')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

for folder in [
    'models/checkpoints',
    'models/loras',
    'models/vae',
    'models/controlnet',
    'models/upscale_models',
    'models/clip',
    'output',
    'input',
    'workflows',
]:
    (DRIVE_ROOT / folder).mkdir(parents=True, exist_ok=True)

print('Persistencia:', DRIVE_ROOT)


## 3. Instalar ComfyUI
Usamos el repositorio oficial y dejamos el código de ComfyUI en `/content` para no hacer que cada operación de la interfaz dependa de la velocidad de Drive. Los modelos y resultados sí viven en Drive.

In [ ]:
COMFY_DIR = Path('/content/ComfyUI')

if not (COMFY_DIR / 'main.py').exists():
    !git clone https://github.com/Comfy-Org/ComfyUI.git /content/ComfyUI

%cd /content/ComfyUI
!pip install -q -r requirements.txt

print('ComfyUI instalado en', COMFY_DIR)


## 4. Conectar modelos y resultados con Google Drive
Creamos enlaces simbólicos para que ComfyUI vea directamente las carpetas persistentes.

In [ ]:
import shutil

MODEL_DIR = COMFY_DIR / 'models'
DRIVE_MODELS = DRIVE_ROOT / 'models'

for name in ['checkpoints','loras','vae','controlnet','upscale_models','clip']:
    target = MODEL_DIR / name
    source = DRIVE_MODELS / name
    source.mkdir(parents=True, exist_ok=True)
    if target.is_symlink() or target.exists():
        if target.is_symlink():
            target.unlink()
        elif target.is_dir():
            shutil.rmtree(target)
    target.symlink_to(source, target_is_directory=True)

output_dir = COMFY_DIR / 'output'
if output_dir.is_symlink():
    output_dir.unlink()
elif output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.symlink_to(DRIVE_ROOT / 'output', target_is_directory=True)

print('Modelos:', DRIVE_MODELS)
print('Outputs:', DRIVE_ROOT / 'output')


## 5. ComfyUI Manager
El Manager permite instalar y administrar custom nodes desde la interfaz.

In [ ]:
MANAGER = COMFY_DIR / 'custom_nodes' / 'ComfyUI-Manager'
if not MANAGER.exists():
    !git clone https://github.com/Comfy-Org/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager
print('Manager listo.')


## 6. Descarga opcional de modelos
Coloca URLs directas de archivos `.safetensors` en las listas. No se descarga ningún modelo automáticamente en esta versión para evitar llenar Drive accidentalmente.

In [ ]:
MODEL_URLS = {
    'checkpoints': [],
    'loras': [],
    'vae': [],
    'controlnet': [],
}

print('Añade URLs a MODEL_URLS cuando quieras automatizar descargas.')


In [ ]:
import urllib.request

def download_urls(category):
    urls = MODEL_URLS.get(category, [])
    destination = DRIVE_MODELS / category
    destination.mkdir(parents=True, exist_ok=True)
    for url in urls:
        filename = url.split('?')[0].rstrip('/').split('/')[-1]
        if not filename:
            raise ValueError(f'No se pudo determinar el nombre: {url}')
        path = destination / filename
        if path.exists():
            print('Ya existe:', path.name)
            continue
        print('Descargando:', filename)
        urllib.request.urlretrieve(url, path)
        print('Listo:', path)

for category in MODEL_URLS:
    download_urls(category)


## 7. Lanzar ComfyUI
La interfaz escucha en el puerto `8188`.

In [ ]:
%cd /content/ComfyUI

import subprocess, threading, time

if 'comfy_process' in globals() and comfy_process.poll() is None:
    print('ComfyUI ya está ejecutándose.')
else:
    comfy_process = subprocess.Popen(
        [sys.executable, 'main.py', '--listen', '0.0.0.0', '--port', '8188'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    def stream_logs():
        for line in comfy_process.stdout:
            print(line, end='')

    threading.Thread(target=stream_logs, daemon=True).start()
    time.sleep(5)
    print('ComfyUI iniciado.')


## 8. URL pública con Cloudflare Tunnel
La URL es temporal y cambia al reiniciar la sesión.

In [ ]:
!wget -q -O /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i /tmp/cloudflared.deb >/dev/null 2>&1 || true

import subprocess, re, time
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

url = None
for _ in range(40):
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(0.25)
        continue
    print(line, end='')
    match = re.search(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        break

print('\nURL de ComfyUI:', url or 'No detectada; revisa los logs.')


## 9. Estructura final
```text
Google Drive/MyDrive/AI-Image/
├── models/
│   ├── checkpoints/
│   ├── loras/
│   ├── vae/
│   ├── controlnet/
│   └── upscale_models/
├── input/
├── output/
└── workflows/
```

### Próximos módulos
- Selector de modelos desde una celda de configuración.
- Descarga segura de checkpoints/LoRAs desde fuentes concretas.
- Workflows preconfigurados para SD 1.5 y SDXL.
- Panel sencillo para prompt, seed, steps, CFG y LoRA weights.
- Detección automática de T4/L4/A100 y ajuste de memoria.
